# Assignment 5: Library Chatbot v2
### AITCD001 Chatbots Development · Learning Outcome 2

**Release:** Wednesday 7 October 2026
**Due:** Monday 12 October 2026, 23:59
**Total:** 100 Points

---

Assignment 4 built a library chatbot that gives every message one tag. This assignment upgrades
the same chatbot with the Week 5 techniques: a **router** of handler functions, **several
requests in one message**, and **memory** across turns. The library, its intents and its test set
are unchanged.

| Part | Work | Points |
|---|---|---|
| A | The router: handlers, a handler that computes, and a new intent | 35 |
| B | Several requests in one message | 25 |
| C | Memory: a name, a follow-up, a side question during a booking | 25 |
| D | Test conversations, written answers and the README | 15 |

**Starting point.** Sections 1 to 3 are provided: the library and a working library chatbot to
build on. Run them first; everything you write starts at Part A.

**Working notes**

- Cells run in order. The chat cell waits for keyboard input: run it on its own.
- A **Sample Output** block sits under each task, produced by a complete solution. Until a task is
  done, the output will differ; that is expected. There are no automated checks.
- Restart the runtime and run everything in order before submitting.

---

## Section 1: Setup

In [1]:
# AITCD001, Assignment 5: Library Chatbot v2
# Name:            TODO
# GitHub username: TODO

import re

print("ready")

ready


**Sample Output**

```
ready
```

## Section 2: The library, provided

The replies, the fallback, the cleaning function, the provided parts of the booking flow and the Assignment 4 test set. Nothing here changes.

In [2]:
# PROVIDED: the library. Nothing here needs changing.

RESPONSES = {
    'hours'    : 'We are open 08:00 to 20:00 Monday to Friday, and 09:00 to 13:00 on Saturday. Closed on Sunday.',
    'borrow'   : 'You may borrow up to 4 books at a time, for 14 days.',
    'renew'    : 'A loan can be renewed twice online, unless another reader has reserved the book.',
    'fines'    : 'Overdue books are 200 RWF per book per day, capped at 5,000 RWF.',
    'printing' : 'Printing is 50 RWF per page in black and white, 200 RWF in colour.',
}

FALLBACK = ("Sorry, I cannot help with that. I can answer questions about opening hours, "
            "borrowing, renewals, fines, study rooms and printing. "
            "Please ask at the desk for anything else.")


def normalise(text):
    """Lower case, trimmed, punctuation removed; colons kept, for times."""
    return re.sub(r"[^\w\s:]", "", text).lower().strip()


# The booking flow: what it accepts at each step
DAYS  = r"\b(mon|tue|wed|thu|fri|sat)\w*\b"
SLOTS = r"\b(morning|afternoon|evening)\b"
SIZE  = r"\b([1-8])\b"

flow = {"step": None, "collected": {}}


def reset_flow():
    flow["step"] = None
    flow["collected"] = {}


def start_booking():
    flow["step"] = "day"
    flow["collected"] = {}
    return "Which day would you like the room? Monday to Saturday."


EXIT_WORDS = ("bye", "goodbye", "exit", "quit", "thanks", "murakoze")

# The Assignment 4 test set, unchanged
TESTS = [
    ('what time do you open', 'hours'),
    ('are you open on saturday', 'hours'),
    ('till what time on friday', 'hours'),
    ('how many books can i take', 'borrow'),
    ('borrowing limit?', 'borrow'),
    ('how long can i keep a book', 'borrow'),
    ('can i renew', 'renew'),
    ('renewing my loan', 'renew'),
    ('i want to extend my book', 'renew'),
    ('how much is the fine', 'fines'),
    ('what are the overdue charges', 'fines'),
    ('i returned it late what now', 'fines'),
    ('i want to book a study room', 'rooms'),
    ('can i reserve a room', 'rooms'),
    ('study room booking', 'rooms'),
    ('do you have printing', 'printing'),
    ('how much per page', 'printing'),
    ('can i photocopy', 'printing'),
    ('WHAT TIME DO YOU OPEN', 'hours'),
    ('opening hours?', 'hours'),
    ('are you open at 18:00', 'hours'),
    ('how mny books', 'borrow'),
    ('renewals', 'renew'),
    ('fine is how much', 'fines'),
    ('booking a room for 3 people', 'rooms'),
    ('printing costs', 'printing'),
    ('can you write my assignment', 'fallback'),
    ('do you sell coffee', 'fallback'),
    ('what is the wifi password', 'fallback'),
    ('asdfgh', 'fallback'),
    ('ni angahe for printing', 'printing'),
    ('do you have a bookshop', 'fallback'),
    ('how many books can i take and what are the fines', 'borrow+fines'),
    ('rnw my bk', 'renew'),
    ('can i extend my deadline', 'fallback'),
]

print(len(RESPONSES), "responses and", len(TESTS), "test messages loaded")

5 responses and 35 test messages loaded


**Sample Output**

```
5 responses and 35 test messages loaded
```

## Section 3: The library chatbot, provided

A working chatbot to build on: the rule table, `match()` and the complete room booking flow. Run it; nothing here changes. Every part from Part A onward builds on it.

In [3]:
# PROVIDED: a working library chatbot to build on. Run this cell; nothing here needs
# changing. The rule table, match() and the complete room booking flow.

RULES = [
    ('rooms'    , r"\bstudy room\b|\bbook a room\b|\breserve a room\b|\broom booking\b|\bbooking a room\b|\bstudy room booking\b|\bbook a study\b"),
    ('hours'    , r"\bopen\w*\b|\bclos\w*\b|\bwhat ?time\b|\btill what time\b|\bhours?\b|\b\d{1,2}:\d{2}\b"),
    ('borrow'   , r"\bborrow\w*\b|\bhow many books?\b|\bhow mny\b|\bhow long can i keep\b|\btake out\b"),
    ('renew'    , r"\brenew\w*\b|\bextend\b"),
    ('fines'    , r"\bfines?\b|\boverdue\b|\bcharges?\b|\blate\b|\bpenalt\w*\b"),
    ('printing' , r"\bprint\w*\b|\bphotocop\w*\b|\bper page\b|\bscan\w*\b"),
]


def match(message):
    """The tag of the first pattern that matches, or "fallback"."""
    text = normalise(message)
    for tag, pattern in RULES:
        if re.search(pattern, text):
            return tag
    return "fallback"


def continue_booking(message):
    """Handle one answer inside the booking flow."""
    text = normalise(message)

    if flow["step"] == "day":
        found = re.search(DAYS, text)
        if found:
            flow["collected"]["day"] = found.group(1)
            flow["step"] = "time"
            return "Which time? Morning, afternoon or evening."
        return "Which day? Monday to Saturday."

    if flow["step"] == "time":
        found = re.search(SLOTS, text)
        if found:
            flow["collected"]["time"] = found.group(1)
            flow["step"] = "size"
            return "How many people? Between 1 and 8."
        return "Morning, afternoon or evening?"

    if flow["step"] == "size":
        found = re.search(SIZE, text)
        if found:
            flow["collected"]["size"] = found.group(1)
            c = flow["collected"]
            reset_flow()
            return (f"A room on {c['day']}, {c['time']}, for {c['size']} people. "
                    "Is that correct?")
        return "How many people? Between 1 and 8."

    return "Something went wrong. Starting again."


for m in ["what time do you open", "can i renew", "i want to book a study room"]:
    print(f"{m!r:32} -> {match(m)}")

'what time do you open'          -> hours
'can i renew'                    -> renew
'i want to book a study room'    -> rooms


**Sample Output**

```
'what time do you open'          -> hours
'can i renew'                    -> renew
'i want to book a study room'    -> rooms
```

---

# Part A: The router  [35 Points]

## Section 4: Handlers

A **handler** is a function that answers one intent. Four of these return a fixed reply; `handle_rooms` starts the booking, which a stored sentence cannot do.

In [11]:
def handle_hours(text):
    return RESPONSES["hours"]

def handle_borrow(text):
    return RESPONSES["borrow"]

def handle_renew(text):
    return RESPONSES["renew"]

def handle_printing(text):
    return RESPONSES["printing"]

def handle_rooms(text):
    if flow["step"] is not None:
        return "Your booking is already under way."
    return start_booking()

**Sample Output**

```
You may borrow up to 4 books at a time, for 14 days.
Which day would you like the room? Monday to Saturday.    step is now: day
Your booking is already under way.
```

## Section 5: A handler that computes its reply

The fine depends on the message: 200 RWF per day per book, never more than 5,000 RWF. `found.group(1)` gives the number of days as text; `int()` turns it into a number.

In [14]:
LATE = r"\b(\d+) days?\b"      # "3 days", "1 day"
FINE_PER_DAY = 200
FINE_CAP = 5000


def handle_fines(text):
    found = re.search(LATE, text)
    if found:
        days = int(found.group(1))
        fine = min(days * FINE_PER_DAY, FINE_CAP)
        return f"For {days} days late, the fine is {fine:,} RWF per book."
    return RESPONSES["fines"]


for m in ["what are the fines", "i am 3 days late what is the fine", "40 days late"]:
    print(f"{m!r:36} -> {handle_fines(m)}")

'what are the fines'                 -> Overdue books are 200 RWF per book per day, capped at 5,000 RWF.
'i am 3 days late what is the fine'  -> For 3 days late, the fine is 600 RWF per book.
'40 days late'                       -> For 40 days late, the fine is 5,000 RWF per book.


**Sample Output**

```
'what are the fines'                 -> Overdue books are 200 RWF per book per day, capped at 5,000 RWF.
'i am 3 days late what is the fine'  -> For 3 days late, the fine is 600 RWF per book.
'40 days late'                       -> For 40 days late, the fine is 5,000 RWF per book.
```

## Section 6: The router, and a new intent

`HANDLERS` maps each intent to its handler. The greeting is added at the end of the rule table, so it never beats a real request.

In [15]:
HANDLERS = {
    "rooms":    handle_rooms,
    "hours":    handle_hours,
    "borrow":   handle_borrow,
    "renew":    handle_renew,
    "fines":    handle_fines,
    "printing": handle_printing,
}


# TASK A3: add the greeting intent. Three things:
#   1. handle_greeting returns "Muraho! I am the Campus Library assistant, an automated
#      service, not a librarian. How can I help?"
#   2. append ("greeting", r"\bhello\b|\bhi\b|\bhey\b|\bmuraho\b|\bmwaramutse\b|\bmwiriwe\b")
#      to RULES, at the END, so a greeting never beats a real request
#   3. add handle_greeting to HANDLERS under "greeting"

def handle_greeting(text):
    return "Muraho! I am the Campus Library assistant, an automated service, not a librarian. How can I help?"

RULES.append(("greeting", r"\bhello\b|\bhi\b|\bhey\b|\bmuraho\b|\bmwaramutse\b|\bmwiriwe\b"))
HANDLERS["greeting"] = handle_greeting

# Every tag the rule table can produce needs a handler.
for tag, pattern in RULES:
    if tag not in HANDLERS:
        print("No handler for:", tag)
print(len(HANDLERS), "intents routed")
print(match("mwaramutse"), "->", HANDLERS.get("greeting", handle_greeting)("mwaramutse"))

7 intents routed
greeting -> Muraho! I am the Campus Library assistant, an automated service, not a librarian. How can I help?


**Sample Output**

```
7 intents routed
greeting -> Muraho! I am the Campus Library assistant, an automated service, not a librarian. How can I help?
```

---

# Part B: Several requests in one message  [25 Points]

## Section 7: Splitting a message

The same connectors as Week 5. A comma is not a connector.

In [16]:
CONNECTORS = r"\band\b|\bbut\b|\balso\b|\bthen\b|[?]"


def split_requests(message):
    pieces = re.split(CONNECTORS, message.lower())
    return [normalise(p) for p in pieces if normalise(p)]


print(split_requests("what time do you open and how much is printing?"))
print(split_requests("can i renew a late book"))

['what time do you open', 'how much is printing']
['can i renew a late book']


**Sample Output**

```
['what time do you open', 'how much is printing']
['can i renew a late book']
```

## Section 8: One intent per request

Inside one request the first rule to match still wins, which settles overlaps such as "can i renew a late book".

In [17]:
def route(message):
    paired = []
    seen = set()
    for piece in split_requests(message):
        tag = match(piece)
        if tag != "fallback" and tag not in seen:
            seen.add(tag)
            paired.append((tag, piece))
    return paired


for m in ["what time do you open and how much is printing",
          "can i renew a late book",
          "asdfgh"]:
    print(f"{m!r:50} -> {route(m)}")

'what time do you open and how much is printing'   -> [('hours', 'what time do you open'), ('printing', 'how much is printing')]
'can i renew a late book'                          -> [('renew', 'can i renew a late book')]
'asdfgh'                                           -> []


**Sample Output**

```
'what time do you open and how much is printing'   -> [('hours', 'what time do you open'), ('printing', 'how much is printing')]
'can i renew a late book'                          -> [('renew', 'can i renew a late book')]
'asdfgh'                                           -> []
```

## Section 9: Joining the answers

`respond()` is provided. The decision is which intent’s reply is a question, because a question must come last.

In [18]:
# TASK B3: which intent's reply is a question the reader must answer? Put its tag in ASKS.
ASKS = ["rooms"]


def respond(found):
    """PROVIDED. Answer every request; a reply that asks a question goes last."""
    answers = []
    questions = []
    for tag, request in found:
        reply = HANDLERS[tag](request)
        if tag in ASKS:
            questions.append(reply)
        else:
            answers.append(reply)
    return " ".join(answers + questions)


print(respond(route("i want to book a study room but first what time do you close")))
reset_flow()

We are open 08:00 to 20:00 Monday to Friday, and 09:00 to 13:00 on Saturday. Closed on Sunday. Which day would you like the room? Monday to Saturday.


**Sample Output**

```
We are open 08:00 to 20:00 Monday to Friday, and 09:00 to 13:00 on Saturday. Closed on Sunday. Which day would you like the room? Monday to Saturday.
```

## Section 10: Measuring the gain

Report both figures in the README, as obtained.

In [19]:
# PROVIDED. The Assignment 4 test set, scored the Assignment 4 way and the Assignment 5 way.
def tags_of(message):
    tags = []
    for tag, request in route(message):
        tags.append(tag)
    if len(tags) == 0:
        return "fallback"
    return "+".join(tags)


a4_score = 0
a5_score = 0
for message, expected in TESTS:
    if match(message) == expected:
        a4_score = a4_score + 1
    if tags_of(message) == expected:
        a5_score = a5_score + 1
    reset_flow()

print(f"One tag per message (Assignment 4): {a4_score} of {len(TESTS)}")
print(f"Routed (Assignment 5):              {a5_score} of {len(TESTS)}")
print()
print("Still failing:")
for message, expected in TESTS:
    got = tags_of(message)
    if got != expected:
        print(f"  {message!r:52} expected {expected:14} got {got}")

One tag per message (Assignment 4): 32 of 35
Routed (Assignment 5):              33 of 35

Still failing:
  'rnw my bk'                                          expected renew          got fallback
  'can i extend my deadline'                           expected fallback       got renew


**Sample Output**

```
One tag per message (Assignment 4): 32 of 35
Routed (Assignment 5):              33 of 35

Still failing:
  'rnw my bk'                                          expected renew          got fallback
  'can i extend my deadline'                           expected fallback       got renew
```

---

# Part C: Memory  [25 Points]

## Section 11: Remembering a name

`context` holds what is known about the reader for the whole conversation. The name rule goes first in the table.

In [ ]:
context = {"name": None, "last_intent": None}


def reset_context():
    context["name"] = None
    context["last_intent"] = None


NAME = r"\bmy name is (\w+)"


def handle_name(text):
    found = re.search(NAME, text)
    if found:
        context["name"] = found.group(1).capitalize()
        return f"Nice to meet you, {context['name']}."
    return "Sorry, I did not catch your name."


RULES.insert(0, ("name", r"\bmy name is\b"))
HANDLERS["name"] = handle_name

print(handle_name("my name is grace"))
print(context)
reset_context()

**Sample Output**

```
Nice to meet you, Grace.
{'name': 'Grace', 'last_intent': None}
```

## Section 12: A follow-up

After a fines answer, "and for 10 days?" has no intent of its own, but the context makes it a fines question.

In [ ]:
def follow_up(text):
    if context["last_intent"] == "fines" and re.search(LATE, text):
        return [("fines", text)]
    return []


context["last_intent"] = "fines"
print("after a fines answer: ", follow_up("and for 10 days"))
context["last_intent"] = "hours"
print("after an hours answer:", follow_up("and for 10 days"))
reset_context()

**Sample Output**

```
after a fines answer:  [('fines', 'and for 10 days')]
after an hours answer: []
```

## Section 13: A side question during a booking

Inside a booking the flow sees each message first. If the flow does not move on, the message is a side question: answer it, then ask the booking question again.

In [ ]:
def reply_to(message):
    """One turn of the library chatbot, version 2."""
    text = normalise(message)

    # 1. The exit, before anything else. Leaving clears both memories.
    if text in EXIT_WORDS:
        name = context["name"]
        reset_flow()
        reset_context()
        if name is not None:
            return f"Thank you, {name}. Murakoze."
        return "Thank you. Murakoze."

    # 2. Inside a booking, the flow sees the message first.
    if flow["step"] is not None:
        step_before = flow["step"]
        flow_reply = continue_booking(message)
        if flow["step"] != step_before:     # the flow moved on: it was an answer
            return flow_reply
        # TASK C3: the message was not an answer. Route it. If nothing was found, return
        # flow_reply (the re-ask). Otherwise answer it with respond(), and if the booking is
        # still running, add flow_reply after the answer so the question is asked again.
        return flow_reply

    # 3. Route the message. If nothing matched, try the context.
    found = route(message)
        if len(found) == 0:
            return flow_reply
        answer = respond(found)
        if flow["step"] is not None:
            return f"{answer} {flow_reply}"
        return answer


def run_conversation(turns):
    """Play a scripted conversation from a clean start, printing both sides."""
    reset_flow()
    reset_context()
    for message in turns:
        print("You:", message)
        print("Bot:", reply_to(message))
    print()


run_conversation(["can i reserve a room", "friday", "what time do you close on friday"])

**Sample Output**

```
You: can i reserve a room
Bot: Which day would you like the room? Monday to Saturday.
You: friday
Bot: Which time? Morning, afternoon or evening.
You: what time do you close on friday
Bot: We are open 08:00 to 20:00 Monday to Friday, and 09:00 to 13:00 on Saturday. Closed on Sunday. Morning, afternoon or evening?
```

---

# Part D: Test conversations and write-up  [15 Points]

## Section 14: The eight test conversations

Their expected behaviour is in the brief and the README template. Two are expected to fail. There is no sample output: what is recorded is what this notebook produced.

In [ ]:
# PART D. The eight test conversations. Read the expected behaviour in the brief (or the
# README template) first, then run this cell and record what actually happens.
CONVERSATIONS = [
    ('Two requests joined by "and"',
     ['what time do you open and how much is printing']),
    ('One request that matches two intents',
     ['can i renew a late book']),
    ('The question goes last',
     ['i want to book a study room but first what time do you close']),
    ('Remembering a name',
     ['hello', 'my name is grace', 'how much is printing', 'bye']),
    ('A follow-up with no intent of its own',
     ['i am 3 days late what is the fine', 'and for 10 days?']),
    ('A side question during a booking',
     ['can i reserve a room', 'friday', 'what time do you close on friday', 'afternoon', '4']),
    ('Two questions with no connecting word',
     ['how many books can i take what are the fines']),
    ('An answer that is also a question',
     ['can i reserve a room', 'saturday', 'evening but what time do you close on saturday']),
]

number = 1
for title, turns in CONVERSATIONS:
    print(f"Conversation {number}: {title}")
    run_conversation(turns)
    number = number + 1

In [ ]:
# PART D. Conversations 9 and 10, written by you. Expected behaviour first, in the README.
MY_TESTS = [
    ("9. A conversation that tests memory", ["first message", "second message"]),
    ("10. A conversation designed to break the chatbot", ["your message"]),
]

for title, turns in MY_TESTS:
    print(title)
    run_conversation(turns)

## Section 15: Written answers

In [ ]:
# PART D. Written answers. Type inside the quotes.
answers = """
1. Which test message from Assignment 4 does routing fix, and which still fail? Why can routing not fix those?
Routing fixes messages containing multiple requests joined by connectors (e.g., "how many books can i take and what are the fines"). Messages like abbreviations ("rnw my bk") or out-of-vocabulary rephrasings ("can i extend my deadline") still fail because regex pattern matching relies on explicit keyword presence rather than semantic understanding.

2. For conversations 7 and 8: the cause of each failure, and the part of the chatbot responsible.
Conversation 7 fails because two questions are written without a connecting word in CONNECTORS, causing `split_requests` to treat them as a single string, failing the regex table. Conversation 8 fails because answering with a side question inside a booking step interferes with the strict expected sequence of the booking flow state machine.

3. Choose one failure. What change would fix it, and could that change break a conversation that currently passes?
To fix conversation 7, we could expand `CONNECTORS` or update `split_requests` to split on implicit boundaries or whitespace heuristics between intents. However, this change could break existing single-request queries that contain multiple words or stop words, incorrectly segmenting them into unintended multi-requests.
"""
print(answers)

## Section 16: Try it

Run the cell below **on its own**. Ask two things at once, give a name, ask a fines follow-up, and ask a side question in the middle of a booking.

In [ ]:
def chat():
    """The full chatbot. Run this cell on its own: it waits for keyboard input."""
    reset_flow()
    reset_context()
    print("Campus Library assistant: an automated service, not a librarian.")
    print("Hours, borrowing, renewals, fines, study rooms and printing. Type 'bye' to finish.")
    print()
    while True:
        message = input("You: ")
        print("Bot:", reply_to(message))
        print()
        if normalise(message) in EXIT_WORDS:
            break


chat()

---

## Before submitting

- [ ] Runtime restarted, all cells run in order, no errors.
- [ ] Sections 1 to 3 run without errors.
- [ ] Every `TODO` replaced.
- [ ] The Part B figures and the Part D results are the ones the code produced, not adjusted.
- [ ] All three written answers completed.
- [ ] `README.md` finished in the `week-5` directory.
- [ ] Repository URL opened in a private browsing window and confirmed visible.

Rename this notebook `Lastname_Firstname_Assignment_5.ipynb`.

**The repository URL is the only item submitted on Google Classroom.** A link that does not open
counts as nothing submitted.

Late work: submitted by Thursday 15 October 2026, 23:59, full marks. Submitted after that and by Monday 19 October 2026, 23:59, minus 10 points out of 100. After Monday 19 October 2026, 23:59 the assignment is closed: missing work receives 0, and the solutions are released the next morning. An incomplete notebook is worth more than no notebook, because marks are given for
the parts that are done.